# Hot Routes

In [9]:
!pip cache purge --quiet

In [10]:
!pip install geopandas==1.1.1 --quiet

In [11]:
import geopandas as gpd
import pandas as pd
import warnings

from pymongo import MongoClient
from shapely.geometry import LineString, mapping

warnings.filterwarnings("ignore")

In [12]:
# If you've forked this repo, change OWNER to your GitHub username.
# REPO and BRANCH will normally stay the same unless you renamed them.
OWNER = "singlestore-cookbook"
REPO = "singlestore-cookbook.github.io"
BRANCH = "refs/heads/main"

BASE_URL = f"https://raw.githubusercontent.com/{OWNER}/{REPO}/{BRANCH}/code/part-ml/crime-analytics-with-singlestore-kai/datasets"

In [13]:
bakerloo_stops_csv_url = f"{BASE_URL}/bakerloo_stops.csv"

bakerloo_stops_df = pd.read_csv(bakerloo_stops_csv_url)

In [14]:
bakerloo_line_geojson_url = f"{BASE_URL}/bakerloo_line.geojson"

bakerloo_line = gpd.read_file(bakerloo_line_geojson_url).to_crs(epsg = 4326)

In [15]:
btp_street_csv_url = f"{BASE_URL}/2024-01-2025-01-btp-street.csv"

crimes_df = pd.read_csv(btp_street_csv_url)

In [16]:
try:
    client = MongoClient(connection_url_kai)
    db = client["hot_routes_db"]
    client.drop_database(db)
    print("Connected to Kai successfully")
except Exception as e:
    print(f"Could not connect to Kai: '{e}'")

Connected to Kai successfully


In [17]:
bakerloo_stops = [
    {
        "station_name": r["stn_name"],
        "line": r["line"],
        "location": {
            "type": "Point",
            "coordinates": [r["stn_lon"], r["stn_lat"]]
        }
    }
    for r in bakerloo_stops_df.to_dict(orient = "records")
]

db["bakerloo_stops"].insert_many(bakerloo_stops)
print(f"Inserted {len(bakerloo_stops)} stations")

Inserted 25 stations


In [18]:
db["bakerloo_stops"].count_documents({})

25

In [19]:
db["bakerloo_stops"].find_one()

{'_id': ObjectId('6a51002d5c22aa45fabd9761'),
 'station_name': 'Marylebone',
 'line': 'bakerloo',
 'location': {'type': 'Point', 'coordinates': [-0.163013, 51.521602]}}

In [20]:
def segments(curve):
    return list(map(LineString, zip(curve.coords[:-1], curve.coords[1:])))

bakerloo_sections = [
    {
        "geometry": {
            "type": "LineString",
            "coordinates": list(seg.coords)
        }
    }
    for curve in bakerloo_line.geometry
    for seg in segments(curve)
]

db["bakerloo_sections"].insert_many(bakerloo_sections)
print(f"Inserted {len(bakerloo_sections)} line segments")

Inserted 24 line segments


In [21]:
db["bakerloo_sections"].count_documents({})

24

In [22]:
db["bakerloo_sections"].find_one()

{'_id': ObjectId('6a5100395c22aa45fabd976b'),
 'geometry': {'type': 'LineString',
  'coordinates': [[-0.316946, 51.581786], [-0.308448, 51.570229]]}}

In [23]:
crimes = [
    {
        "crime_id": row.get("Crime ID"),
        "month": row.get("Month"),
        "reported_by": row.get("Reported by"),
        "falls_within": row.get("Falls within"),
        "geometry": {
            "type": "Point",
            "coordinates": [row["Longitude"], row["Latitude"]]
        },
        "location": row.get("Location"),
        "lsoa_code": row.get("LSOA code"),
        "lsoa_name": row.get("LSOA name"),
        "crime_type": row.get("Crime type"),
        "last_outcome_category": row.get("Last outcome category"),
        "context": row.get("Context")
    }
    for _, row in crimes_df.iterrows()
]

db["crimes"].insert_many(crimes)
print(f"Inserted {len(crimes)} crimes")

Inserted 44644 crimes


In [24]:
db["crimes"].count_documents({})

44644

In [25]:
db["crimes"].find_one()

{'_id': ObjectId('6a5100405c22aa45fabd97bd'),
 'crime_id': nan,
 'month': '2024-01',
 'reported_by': 'British Transport Police',
 'falls_within': 'British Transport Police',
 'geometry': {'type': 'Point', 'coordinates': [-0.185454, 51.617878]},
 'location': 'On or near Woodside Park (Lu Station)',
 'lsoa_code': 'E01000277',
 'lsoa_name': 'Barnet 012B',
 'crime_type': 'Other theft',
 'last_outcome_category': nan,
 'context': nan}

In [26]:
bakerloo_line_buffer = bakerloo_line.buffer(0.005)

bakerloo_line_buff = [
    {
        "geometry": mapping(poly)
    }
    for poly in bakerloo_line_buffer
]

db["bakerloo_line_buff"].insert_many(bakerloo_line_buff)
print(f"Inserted {len(bakerloo_line_buff)} line buffer polygons")

Inserted 1 line buffer polygons


In [27]:
db["bakerloo_line_buff"].count_documents({})

1

In [28]:
db["bakerloo_line_buff"].find_one()

{'_id': ObjectId('6a5100455c22aa45fabe45e6'),
 'geometry': {'type': 'Polygon',
  'coordinates': [[[-0.31443398617096996, 51.58610916857441],
    [-0.31400443053839167, 51.5858291632545],
    [-0.31360581092362083, 51.58550663663021],
    [-0.31324231954569604, 51.585144980662704],
    [-0.31291777918557934, 51.58474799883023],
    [-0.30441977918557933, 51.57319099883023],
    [-0.30412765171710754, 51.57274586128233],
    [-0.2997902003342216, 51.56530035732638],
    [-0.29343056457613337, 51.556422824962034],
    [-0.2740086379053138, 51.548686014544494],
    [-0.2738925638531693, 51.54863807829827],
    [-0.25616593113642977, 51.54105537270265],
    [-0.24210840226931313, 51.53748673268892],
    [-0.22525071648315476, 51.535598739476846],
    [-0.20583082285512747, 51.539352148112414],
    [-0.20513332509409826, 51.53943667957493],
    [-0.19448332509409824, 51.53997267957493],
    [-0.19398282805554815, 51.53997278747466],
    [-0.1934848276929434, 51.53992285816378],
    [-0.19299